# Kissan AI - Crop Disease Model Training (ResNet-18)
Welcome to the Colab training environment! Make sure to set the runtime to GPU (Runtime > Change runtime type > T4 GPU).
Upload your `plant_disease_dataset.zip` file using the folder icon on the left, and then run these cells!

In [ ]:
!unzip -q plant_disease_dataset.zip -d data/

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import os

print('PyTorch version:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
# Data Augmentation and Normalization
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

data_dir = 'data/plant_disease_dataset'  # Directory extracted from ZIP

image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x),
                                          data_transforms[x])
                  for x in ['train', 'test']}
dataloaders = {x: DataLoader(image_datasets[x], batch_size=32,
                                             shuffle=True, num_workers=2)
              for x in ['train', 'test']}
dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'test']}
class_names = image_datasets['train'].classes

print('Classes:', class_names)
print('Train size:', dataset_sizes['train'])
print('Test size:', dataset_sizes['test'])

In [ ]:
# Load Pre-trained ResNet-18
import torchvision.models as models
model_ft = models.resnet18(weights='IMAGENET1K_V1')

# Freeze early layers for transfer learning
for param in model_ft.parameters():
    param.requires_grad = False

# Replace final layer to match our number of classes
num_ftrs = model_ft.fc.in_features
model_ft.fc = nn.Linear(num_ftrs, len(class_names))

model_ft = model_ft.to(device)

criterion = nn.CrossEntropyLoss()
# Optimize only the final layer
optimizer_ft = optim.Adam(model_ft.fc.parameters(), lr=0.001)

In [ ]:
import time
import copy

def train_model(model, criterion, optimizer, num_epochs=10):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        for phase in ['train', 'test']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'test' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
        print()  # Add an empty line between epochs

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best test Acc: {best_acc:4f}')

    model.load_state_dict(best_model_wts)
    return model

# Train for 5 epochs. This should easily hit >85% accuracy with ResNet-18!
model_ft = train_model(model_ft, criterion, optimizer_ft, num_epochs=5)

In [ ]:
# Save the model weights and the class names
torch.save({
    'state_dict': model_ft.state_dict(),
    'class_names': class_names
}, 'kissan_disease_model.pth')
print('Model saved to kissan_disease_model.pth!')